# ESR of two weakly coupled spins

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const

from spinfinity import ESR, SpinSys, bath, hamil
from spinfinity import helperfunctions as hf

%load_ext autoreload
%autoreload 2

# Constructing the spin Hamiltonian

The two spin-1/2 experience an external field of 0.5 T along z.
The tip field acts only on the readout spin.

The exchange interaction is parameterized relative to the z axis:

$$
H_{\mathrm{ex}}
=
J_\perp(S_{0x}S_{1x}+S_{0y}S_{1y})
+
J_\parallel S_{0z}S_{1z}.
$$

We compare two longitudinal tip fields and then calculate a tip-field map.
The spectra use thermal populations

In [ ]:
# Define the spin system
spin = SpinSys([0.5, 0.5])

spin.B = [0.0, 0.0, 0.5]  # T
spin.BTip = [1e-9, 0.0, 0.0]  # T
spin.gfactorvector = [2.0, 2.0]
spin.ReadoutSpin = 0
spin.tip = "f"
spin.T = 1.0  # K

# Exchange parameters expressed as energy / h in GHz
J_perp_GHz = 0.4
J_parallel_GHz = 0.2

# Convert to the meV units required by the Hamiltonian
exchange_meV = np.array([
    J_perp_GHz,
    J_perp_GHz,
    J_parallel_GHz,
]) * spin.GHztomeVConversion

hamil.assign_ExchangeInteraction(
    spin,
    [(0, 1, exchange_meV)],
)

# Do not add a separate dipolar interaction
spin.DipoleVector[:] = 0.0



## Eigenstates

A small longitudinal tip field leaves the two spins nearly resonant,
allowing exchange to mix the antiparallel product states. A larger tip
field detunes the readout spin and reduces this mixing.

The x component remains fixed at 1e-9 T.

In [ ]:
Btip_z_values = [0.1e-3, 30e-3]  # T

for Btip_z in Btip_z_values:
    spin.BTip = [1e-9, 0.0, Btip_z]

    hamil.build_Hamiltonian(spin)
    hamil.calc_EigEnergies(spin)
    hamil.calc_EigStates(spin)

    print(f"Btip,z = {Btip_z * 1e3:.1f} mT")
    print("Relative energies (GHz):", spin.E_All_inGHz)

    eigen_result = hamil.plot_EigenMatrix(spin, plot_type="p")

## ESR spectra at two tip fields

Calculate phenomenological spectra with a prescribed FWHM of 20 MHz.
Each spectrum is normalized separately, so the comparison concerns peak
positions and relative peak weights, not absolute intensities.

In [ ]:
freq_array = np.linspace(12.5, 15.5, 2001)  # GHz
lw = 0.02  # GHz; FWHM

# Initialize Signals array to store the ESR signals for each Btip_z value
Signals = np.zeros((len(Btip_z_values), len(freq_array)))

for i,Btip_z in enumerate(Btip_z_values):
    spin.BTip = [1e-9, 0.0, Btip_z]

    hamil.build_Hamiltonian(spin)
    hamil.calc_EigEnergies(spin)
    hamil.calc_EigStates(spin)
    freqs, signal = ESR.calc_FreqSweep(
        spin,
        freq_array=freq_array,
        AllowPumping=False,
        lw=lw,
        norm=True,
        output=False,
    )
    Signals[i] = signal

# Create Plot
fig, ax = plt.subplots(figsize=(8, 4.5))

for Btip_z, signal in zip(Btip_z_values, Signals):
    ax.plot(
        freq_array,
        signal,
        label=f"$B_{{\\mathrm{{Tip}},z}} = {Btip_z * 1e3:.1f}$ mT",
    )

ax.set_xlabel("Frequency (GHz)")
ax.set_ylabel("Normalized ESR signal")
ax.set_title("ESR spectra for two longitudinal tip fields")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## Tip-field map at theta = 0

Sweep the longitudinal tip field with `Angle=0`. The function retains
a small transverse x component. Each spectrum is normalized separately.

The right axis represents the bare local Zeeman detuning, not the shift
of each coupled-spin resonance. An even number of field points avoids
interpreting numerical residuals at exactly zero longitudinal tip field.

In [ ]:
BTip_array = np.linspace(-0.04, 0.04, 402)  # T

map_result = ESR.plot_Tipfieldmap(
    spin,
    freq_array=freq_array,
    BTip_array=BTip_array,
    lw=lw,
    norm=True,
    AllowPumping=False,
    Angle=0.0,
)